In [1]:
#!pip install adversarial-robustness-toolbox
import numpy as np
import tensorflow as tf
from keras.datasets import cifar10
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from art.attacks.evasion import SaliencyMapMethod
from art.estimators.classification  import KerasClassifier
from sklearn.metrics import accuracy_score

tf.compat.v1.disable_eager_execution()

import tensorflow as tf
import json
# download mnist data and split into train and test sets
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()
# reshape data to fit model
X_train, X_test = X_train/255, X_test/255
# one-hot encode target column
y_train = tf.keras.utils.to_categorical(y_train)
y_test = tf.keras.utils.to_categorical(y_test)
# create model
model = Sequential()
model.add(Conv2D(32, (3, 3), padding='same', input_shape=(32, 32, 3), activation='relu'))
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(10, activation='softmax'))


# compile model using accuracy as a measure of model performance
model.compile(optimizer='adam', loss='categorical_crossentropy',
              metrics=['accuracy'])

# train model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=5)

json.dump({'model': model.to_json()}, open("model.json", "w"))
model.save_weights("model_weights.h5")


#Create a KerasClassifier for the model
classifier = KerasClassifier(model=model, clip_values=(0, 1),  use_logits=False)
# Generate adversarial examples
x_test = X_test # your test data
y = y_test # the true labels for the test data
jsma = SaliencyMapMethod(classifier=classifier, theta=1, gamma=0.3)
#x_test_adv = jsma.generate(x=x_test, y=y)
x_test_adv = jsma.generate(x_test)

# Evaluate the model's accuracy before adv attack on the test dataset
score1 = model.evaluate(x_test, y_test, verbose=0)
print('Test accuracy:', score1[1])

# Evaluate the model's accuracy after adv attack on the test dataset
model.fit(X_train, y_train, batch_size=32, epochs=5, validation_data=(x_test_adv, y_test))
score1 = model.evaluate(x_test_adv, y_test, verbose=0)
print('Test accuracy after adversarial aatack:', score1[1])

x_test_adv
x_test
print(f"X_train shape: {x_test_adv.shape}")
print(f"y_train shape: {x_test.shape}")



Train on 50000 samples, validate on 10000 samples
Epoch 1/5
50000/50000 [==============================] - ETA: 0s - loss: 1.5066 - accuracy: 0.4490

C:\Users\nidhi.jain1\Anaconda3\lib\site-packages\keras\engine\training_v1.py:2333: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates


50000/50000 [==============================] - 543s 11ms/sample - loss: 1.5066 - accuracy: 0.4490 - val_loss: 1.1362 - val_accuracy: 0.5917
Epoch 2/5
50000/50000 [==============================] - 486s 10ms/sample - loss: 1.1125 - accuracy: 0.6041 - val_loss: 0.9097 - val_accuracy: 0.6785
Epoch 3/5
35264/50000 [====================>.........] - ETA: 2:15 - loss: 0.9657 - accuracy: 0.6600

KeyboardInterrupt: 